# F5A_CV — F5a Part B: offset diagnosis on the 48 CV checkpoints (GAE, MLP-AE; partitions 0–5) (Track F, chat D36-F)

**Design:** `results/r4_f5a/DESIGN_F5A.md` §10 (approved 21/11/2026 12:43). Inference only, the committed R2_A / R3
checkpoints (in the repository); no training. **Runs a SMOKE first** (GAE and MLP-AE fold 1 of partition 0, every gate,
measured) and STOPs before the full run if anything differs from the committed rows. **Accelerator:** None (CPU).
**Internet:** ON (pip, git clone). **Input (Add Input):** `thesis-r2-chbmit-features`. **Output:**
`/kaggle/working/r4/f5a_cv` (→ `results/r4_f5a/cv`). **Runtime:** ≈ 0.4–0.6 h (computed; the smoke writes
`projection.json`). Code: `github.com/Nhanvas/Thesis` ref `main`; every module's sha256 is asserted below.
**Nothing printed here is a research result.** Do not edit; *Save Version → Save & Run All (Commit)*.

In [ ]:
# SETUP: packages + repository (read only)
import os, sys, subprocess, time, json, hashlib
T_NB = time.time()
NOTEBOOK, REPO_REF = "F5A_CV", "main"
TEST = bool(os.environ.get("F5A_TEST_REPO"))                     # synthetic test in the build container only
WORK = os.environ.get("F5A_TEST_WORK", "/kaggle/working")
INPUT = os.environ.get("F5A_TEST_INPUT", "/kaggle/input")
SCRATCH = os.environ.get("F5A_TEST_SCRATCH", "/tmp/r2")
if TEST:
    REPO = os.environ["F5A_TEST_REPO"]; HEAD = "test"
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch_geometric==2.7.0", "ruptures==1.1.10", "timescoring==0.0.7"])
    REPO = "/tmp/Thesis"
    if not os.path.exists(REPO + "/src/r4_f5a_cv.py"):
        subprocess.check_call(f"rm -rf {REPO} && git clone -q https://github.com/Nhanvas/Thesis.git {REPO} "
                              f"&& cd {REPO} && git checkout -q {REPO_REF}", shell=True)
    HEAD = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"]).decode().strip()
print("repository", REPO, HEAD, f"({time.time()-T_NB:.0f}s)")

In [ ]:
# START: sha256 of every module (the reviewed code only), environment, imports
WANT = {"r4_f5a_cv.py": "559d460958a5b080", "r2_r2.py": "542968bb851bc05c", "r2_core.py": "902d15ad4feb3e8a", "f1_core.py": "bf4bfd69d7669226", "model_cmp_core.py": "179b91278035b37a", "v9_e1_baselines.py": "39daafbd63a7e84a", "p3_levers.py": "feb247d2d08c52ba", "p2_scalars.py": "11feeaf3ba95ced4", "e18_cpd_localizer.py": "e18c0ffe37f78e77", "e10_kernel_cpd.py": "5c692e33d92fd0fb", "e16_recording_baseline.py": "f170e1c59f5b0447", "e17b_latent_family.py": "3b4f034dca00eaf3", "retrain/gae_joint.py": "06b0566c97cd0ec4"}
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]
bad = {n: sha(f"{REPO}/src/{n}") for n in WANT if sha(f"{REPO}/src/{n}") != WANT[n]}
assert not bad, f"modules differ from the reviewed code (rebuild the notebook or pin the right commit): {bad}"
os.environ.update(THESIS_REPO=REPO, R2_WORK=WORK, R2_INPUT=INPUT, R2_SCRATCH=SCRATCH, OMP_NUM_THREADS="1",
                  OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1")
if TEST:
    os.environ["R2_TEST"] = "1"
sys.path[:0] = [REPO + "/src"]
import r4_f5a_cv as M
assert M.__file__ == f"{REPO}/src/r4_f5a_cv.py", M.__file__
print("CPU cores:", os.cpu_count(), "| r4_f5a_cv", M.CODE_SHA)

In [ ]:
# RUN: G-data, D0', SMOKE (G-k, G-rows, G-eng, G-seg, G-worker-sha, projection), then the full run
ERR = None
try:
    M.main(NOTEBOOK, HEAD, workers=int(os.environ.get("F5A_WORKERS", "0")) or None)
except SystemExit as e:
    ERR = str(e)

In [ ]:
# END: gates, projection, outputs, measured runtime
OUT = os.path.join(WORK, "r4", "f5a_cv")
for f in ("gates.json", "smoke/gates.json", "projection.json", "run_meta.json"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        print("=====", f); print(open(p).read()[:4000])
for root, _, fs in os.walk(OUT):
    for f in sorted(fs):
        p = os.path.join(root, f); print(os.path.relpath(p, WORK), os.path.getsize(p))
print(f"MEASURED notebook runtime: {(time.time() - T_NB) / 3600:.3f} h")
if ERR:
    raise RuntimeError(f"STOPPED: {ERR}")